# Итоговая работа по статистике в Python

Для работы используется датасет `Global YouTube Statistics.csv`.


Описание датасета с переводом.

| Столбец | Описание |
|---|---|
| `rank` | Позиция YouTube-канала в рейтинге по количеству подписчиков |
| `Youtuber` | Название YouTube-канала / имя ютубера |
| `subscribers` | Количество подписчиков |
| `video views` | Общее количество просмотров всех видео канала |
| `category` | Категория или тематика канала |
| `Title` | Название YouTube-канала |
| `uploads` | Количество загруженных видео |
| `Country` | Страна происхождения канала |
| `Abbreviation` | Сокращённое обозначение страны |
| `channel_type` | Тип канала, например индивидуальный или брендовый |
| `video_views_rank` | Позиция канала в рейтинге по общему количеству просмотров |
| `country_rank` | Позиция канала в рейтинге по количеству подписчиков внутри своей страны |
| `channel_type_rank` | Позиция канала в рейтинге среди каналов того же типа |
| `video_views_for_the_last_30_days` | Общее количество просмотров за последние 30 дней |
| `lowest_monthly_earnings` | Минимальная оценка месячного дохода канала |
| `highest_monthly_earnings` | Максимальная оценка месячного дохода канала |
| `lowest_yearly_earnings` | Минимальная оценка годового дохода канала |
| `highest_yearly_earnings` | Максимальная оценка годового дохода канала |
| `subscribers_for_last_30_days` | Количество новых подписчиков за последние 30 дней |
| `created_year` | Год создания канала |
| `created_month` | Месяц создания канала |
| `created_date` | Точная дата создания канала |
| `Gross tertiary education enrollment (%)` | Доля населения страны, охваченного высшим образованием, % |
| `Population` | Численность населения страны |
| `Unemployment rate` | Уровень безработицы в стране |
| `Urban_population` | Численность городского населения |
| `Latitude` | Географическая широта страны |
| `Longitude` | Географическая долгота страны |


## Задание 1. EDA

Проведите разведывательный анализ данных.

1. Задайте не менее 6 вопросов к данным и постарайтесь ответить на них, используя визуализацию.
2. Изобразите матрицу корреляций для нахождения взаимосвязей.


In [ ]:
# Импорты

import importlib.util
import subprocess
import sys

if importlib.util.find_spec('missingno') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'missingno'])

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from scipy.stats import pearsonr, spearmanr, probplot

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier


In [ ]:
# Загрузка датасета

df = pd.read_csv('/content/Global YouTube Statistics.csv', encoding='latin1')

df.head()


In [ ]:
# Сохраним исходную копию данных

df_raw = df.copy()


In [ ]:
# Размер таблицы

df.shape


In [ ]:
# Типы данных и количество непустых значений

df.info()


In [ ]:
# Проверка полных дубликатов

df.duplicated().sum()


В таблице 995 строк и 28 столбцов. Полных дубликатов нет. Перед дальнейшим анализом отдельно посмотрим на пропуски и несколько подозрительных значений.


In [ ]:
# Проверим подозрительные годы создания каналов

df[df['created_year'] < 2005][
    ['Youtuber', 'created_year', 'created_month', 'created_date']
]


Для канала `YouTube` обнаружена техническая дата `01.01.1970`. Она не соответствует реальному времени создания канала, поэтому заменяем только поля даты на пропуски, не удаляя всю строку.


In [ ]:
mask = (df['Youtuber'] == 'YouTube') & (df['created_year'] == 1970)

df.loc[
    mask,
    ['created_year', 'created_month', 'created_date']
] = pd.NA


### Пропуски

Сначала оценим количество пропусков численно, а затем посмотрим на их структуру с помощью библиотеки `missingno`.


In [ ]:
missing = pd.DataFrame({
    'Количество пропусков': df.isna().sum(),
    'Доля пропусков, %': (df.isna().mean() * 100).round(2)
})

missing[missing['Количество пропусков'] > 0].sort_values(
    'Количество пропусков',
    ascending=False
)


In [ ]:
# Визуализация структуры пропусков

msno.matrix(
    df,
    figsize=(14, 6),
    sparkline=False,
    fontsize=9
)

plt.title('Структура пропусков в датасете')
plt.show()


Наибольшая доля пропусков приходится на `subscribers_for_last_30_days`. Также заметен общий блок пропусков в признаках, связанных со страной. Вместо удаления всех таких строк попробуем восстановить часть значений там, где между признаками есть логическая связь.


In [ ]:
# Сохраним количество пропусков до точечного восстановления

linked_columns = [
    'Country',
    'Abbreviation',
    'country_rank',
    'channel_type',
    'channel_type_rank'
]

missing_before = df[linked_columns].isna().sum()


def fill_from_unique_mapping(data, target, key):
    """Заполняет target по key только там, где key однозначно соответствует одному target."""
    known = data[[key, target]].dropna()
    counts = known.groupby(key)[target].nunique()
    safe_keys = counts[counts == 1].index

    mapping = (
        known[known[key].isin(safe_keys)]
        .drop_duplicates(subset=key)
        .set_index(key)[target]
    )

    data[target] = data[target].fillna(data[key].map(mapping))


def strict_mode(series):
    """Возвращает моду только если она действительно единственная."""
    counts = series.dropna().value_counts()

    if counts.empty:
        return pd.NA

    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return pd.NA

    return counts.index[0]


# Country <-> Abbreviation: используем только однозначные соответствия
fill_from_unique_mapping(df, 'Abbreviation', 'Country')
fill_from_unique_mapping(df, 'Country', 'Abbreviation')

# Country <- country_rank только там, где такой rank однозначно связан с одной страной
fill_from_unique_mapping(df, 'Country', 'country_rank')
fill_from_unique_mapping(df, 'Abbreviation', 'Country')

# country_rank: заполняем модой внутри страны только при наличии единственной моды
country_rank_fill = df.groupby('Country')['country_rank'].transform(strict_mode)
df['country_rank'] = df['country_rank'].fillna(country_rank_fill)

# channel_type <- channel_type_rank только при однозначном соответствии
fill_from_unique_mapping(df, 'channel_type', 'channel_type_rank')

# channel_type_rank: аналогично заполняем модой внутри типа канала
channel_type_rank_fill = df.groupby('channel_type')['channel_type_rank'].transform(strict_mode)
df['channel_type_rank'] = df['channel_type_rank'].fillna(channel_type_rank_fill)

missing_after = df[linked_columns].isna().sum()

pd.DataFrame({
    'До заполнения': missing_before,
    'После заполнения': missing_after
})


Здесь используется осторожное точечное восстановление: значение заполняется только при однозначной связи или при наличии единственной моды внутри группы. Если соответствие неоднозначно, пропуск сохраняется, чтобы не добавлять в данные вымышленные значения.


In [ ]:
# Описательная статистика уже после базовой очистки

df.describe().T


In [ ]:
# Проверка важных числовых признаков на нули

columns_to_check = [
    'subscribers',
    'video views',
    'uploads',
    'video_views_for_the_last_30_days'
]

for col in columns_to_check:
    print(col, (df[col] == 0).sum())


У многих числовых признаков большой разброс: есть несколько очень крупных каналов, которые сильно отличаются от основной массы. В `video views` есть 8 нулевых значений, а в `uploads` — 43. Пока оставляем их и фильтруем только там, где этого требует задание.


In [ ]:
# Проверка уникальных категорий

print('Количество уникальных категорий:', df['category'].nunique())
print()
print(df['category'].value_counts(dropna=False))


Распределение по категориям неравномерное: несколько категорий представлены очень широко, а часть встречается редко. Кроме того, есть записи с пропущенной категорией.


In [ ]:
# Проверка уникальных стран

print('Количество уникальных стран:', df['Country'].nunique())
print()
print(df['Country'].value_counts(dropna=False).head(15))


Распределение по странам также неравномерное. Часть значений `Country` остаётся пропущенной даже после точечного восстановления, потому что для них не удалось найти однозначное соответствие.


### Исследовательские вопросы


**Вопрос 1. Какие категории YouTube-каналов наиболее распространены в датасете?**


In [ ]:
category_counts = df['category'].value_counts()

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    x=category_counts.values,
    y=category_counts.index
)

ax.bar_label(ax.containers[0], padding=3)

plt.title('Количество YouTube-каналов по категориям')
plt.xlabel('Количество каналов')
plt.ylabel('Категория')
plt.show()


Больше всего каналов относится к `Entertainment`, `Music` и `People & Blogs`. Распределение заметно неравномерное: несколько категорий занимают большую часть выборки, а некоторые встречаются всего несколько раз.


**Вопрос 2. Какие страны представлены наибольшим количеством YouTube-каналов?**


In [ ]:
country_counts = df['Country'].value_counts().head(10)

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    x=country_counts.values,
    y=country_counts.index
)

ax.bar_label(ax.containers[0], padding=3)

plt.title('Топ-10 стран по количеству YouTube-каналов')
plt.xlabel('Количество каналов')
plt.ylabel('Страна')
plt.show()


Больше всего каналов представлено из США и Индии, после них идёт большой отрыв. Значит, эти две страны заметно сильнее остальных представлены в данном датасете.


**Вопрос 3. Есть ли связь между количеством просмотров и количеством подписчиков?**


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='video views',
    y='subscribers'
)

plt.title('Связь между количеством просмотров и подписчиков')
plt.xlabel('Количество просмотров')
plt.ylabel('Количество подписчиков')
plt.show()


На графике видна положительная связь: чем больше у канала просмотров, тем обычно больше и подписчиков. При этом есть несколько очень крупных каналов-выбросов, поэтому для количественной оценки связи важно учитывать не только линейную корреляцию Пирсона, но и более устойчивую ранговую корреляцию.


**Вопрос 4. Связано ли количество загруженных видео с общим количеством просмотров?**


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='uploads',
    y='video views'
)

plt.title('Связь между количеством загруженных видео и просмотрами')
plt.xlabel('Количество загруженных видео')
plt.ylabel('Количество просмотров')
plt.show()


Явной связи между количеством загруженных видео и просмотрами не видно. Большое число публикаций само по себе не означает, что канал будет получать больше просмотров.


**Вопрос 5. Различается ли количество подписчиков между наиболее популярными категориями каналов?**


In [ ]:
top_categories = df['category'].value_counts().head(5).index
df_top_categories = df[df['category'].isin(top_categories)]

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_top_categories,
    x='category',
    y='subscribers'
)

plt.title('Распределение количества подписчиков по популярным категориям')
plt.xlabel('Категория')
plt.ylabel('Количество подписчиков')
plt.xticks(rotation=45)
plt.show()


Медианное количество подписчиков у пяти крупнейших категорий довольно похожее. Главное отличие — разброс: в некоторых категориях есть отдельные каналы с очень большой аудиторией.


**Вопрос 6. Как различается количество просмотров между наиболее популярными категориями каналов?**


In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_top_categories,
    x='category',
    y='video views'
)

plt.title('Распределение количества просмотров по популярным категориям')
plt.xlabel('Категория')
plt.ylabel('Количество просмотров')
plt.xticks(rotation=45)
plt.show()


По просмотрам категория `Music` выглядит выше части остальных категорий, но внутри каждой группы разброс большой. Одна только категория довольно слабо описывает, сколько просмотров будет у конкретного канала.


### Форма распределений и корреляционная матрица

По графикам уже видно много выбросов и выраженную асимметрию. Дополнительно посмотрим коэффициенты асимметрии для основных числовых показателей.


In [ ]:
distribution_columns = [
    'subscribers',
    'video views',
    'uploads',
    'video_views_for_the_last_30_days',
    'subscribers_for_last_30_days'
]

df[distribution_columns].skew().sort_values(ascending=False).to_frame('Асимметрия')


Поскольку распределения заметно отличаются от нормальных и содержат выбросы, для общей матрицы взаимосвязей используем коэффициент Спирмена. Он основан на рангах и меньше зависит от экстремальных значений, чем коэффициент Пирсона.


In [ ]:
corr_columns = [
    'subscribers',
    'video views',
    'uploads',
    'video_views_for_the_last_30_days',
    'lowest_monthly_earnings',
    'highest_monthly_earnings',
    'lowest_yearly_earnings',
    'highest_yearly_earnings',
    'subscribers_for_last_30_days',
    'created_year',
    'Gross tertiary education enrollment (%)',
    'Population',
    'Unemployment rate',
    'Urban_population'
]

corr_matrix_spearman = df[corr_columns].corr(method='spearman')

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr_matrix_spearman,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)

plt.title('Матрица ранговых корреляций Спирмена')
plt.tight_layout()
plt.show()


Матрица Спирмена показывает монотонные взаимосвязи без сильной зависимости от выбросов. Связь между просмотрами и подписчиками остаётся заметной, но ранговая оценка оказывается более сдержанной, чем коэффициент Пирсона. Показатели предполагаемого дохода по-прежнему очень тесно связаны между собой, а большинство социально-экономических показателей страны с основными характеристиками каналов связаны слабее.


## Задание 2. Корреляционный анализ и линейная регрессия

По условию задания необходимо рассчитать коэффициент Пирсона для `subscribers` и `video views`, удалить строки с нулевыми просмотрами, построить линейную регрессию и оценить её на тестовой выборке.

Так как EDA показал выраженную асимметрию распределений, дополнительно рассчитаем коэффициент Спирмена и сравним результаты.


In [ ]:
# Оставляем два нужных столбца

df_corr = df[['subscribers', 'video views']].copy()

# Убираем строки с нулевыми просмотрами
df_corr = df_corr[df_corr['video views'] != 0]

df_corr.shape


In [ ]:
# Корреляция Пирсона — по условию задания

r_pearson, p_pearson = pearsonr(
    df_corr['video views'],
    df_corr['subscribers']
)

# Корреляция Спирмена — более устойчивая для текущих распределений
r_spearman, p_spearman = spearmanr(
    df_corr['video views'],
    df_corr['subscribers']
)

print('Пирсон:')
print('r =', round(r_pearson, 3))
print('p-value =', p_pearson)

print('\nСпирмен:')
print('rho =', round(r_spearman, 3))
print('p-value =', p_spearman)


По условию задания коэффициент Пирсона равен примерно `0.824`, то есть линейная связь выглядит сильной. Однако коэффициент Спирмена составляет примерно `0.583` и показывает более умеренную монотонную связь. Такая разница хорошо согласуется с EDA: в данных есть сильная асимметрия и крупные выбросы, которые заметно влияют на Пирсона. Поэтому для содержательной интерпретации этих данных ранговая корреляция Спирмена выглядит надёжнее, а Пирсон оставляем как обязательную часть задания.


### Линейная регрессия


In [ ]:
X = df_corr[['video views']]
y = df_corr['subscribers']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=13
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = model.score(X_test, y_test)

print('R² на тестовой выборке:', round(r2, 3))
print('Коэффициент:', model.coef_[0])
print('Свободный член:', model.intercept_)


На тестовой выборке `R²` составляет около `0.737`. Это значит, что модель с одним признаком `video views` объясняет значительную часть разброса количества подписчиков, но не учитывает все различия между каналами.


In [ ]:
# Визуализация линейной регрессии

plt.figure(figsize=(10, 6))

plt.scatter(
    X_test['video views'],
    y_test,
    alpha=0.6,
    label='Фактические значения'
)

order = X_test['video views'].argsort()

plt.plot(
    X_test['video views'].iloc[order],
    y_pred[order],
    label='Линейная регрессия'
)

plt.title('Линейная регрессия: просмотры и подписчики')
plt.xlabel('Количество просмотров')
plt.ylabel('Количество подписчиков')
plt.legend()
plt.show()


Линия регрессии хорошо показывает общий восходящий тренд, но отдельные каналы заметно от неё отклоняются. Поэтому одного `R²` недостаточно: дополнительно посмотрим на остатки модели.


### Диагностика остатков


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(9, 5))

sns.histplot(
    residuals,
    bins=30,
    kde=True
)

plt.axvline(0, linestyle='--')
plt.title('Распределение остатков линейной регрессии')
plt.xlabel('Остаток')
plt.ylabel('Количество наблюдений')
plt.show()

print('Среднее остатков:', round(residuals.mean(), 2))
print('Асимметрия остатков:', round(residuals.skew(), 3))


In [ ]:
# Q-Q график для дополнительной проверки нормальности остатков

plt.figure(figsize=(7, 7))
probplot(residuals, dist='norm', plot=plt)
plt.title('Q-Q график остатков')
plt.show()


В идеальной ситуации остатки должны быть примерно симметричны относительно нуля, а точки на Q-Q графике — располагаться близко к прямой. Если заметны асимметрия, длинные хвосты или сильные отклонения на концах Q-Q графика, предпосылка нормальности остатков выполняется плохо. Для этого датасета такая проверка особенно важна из-за крупных выбросов, обнаруженных ещё на этапе EDA.


### Что изменится, если не удалять нулевые просмотры?


In [ ]:
df_corr_with_zero = df[['subscribers', 'video views']].copy()

X_zero = df_corr_with_zero[['video views']]
y_zero = df_corr_with_zero['subscribers']

X_train_zero, X_test_zero, y_train_zero, y_test_zero = train_test_split(
    X_zero,
    y_zero,
    test_size=0.2,
    random_state=13
)

model_zero = LinearRegression()
model_zero.fit(X_train_zero, y_train_zero)

r2_zero = model_zero.score(X_test_zero, y_test_zero)

print('R² без нулевых просмотров:', round(r2, 3))
print('R² с нулевыми просмотрами:', round(r2_zero, 3))
print('Изменение R²:', round(r2_zero - r2, 3))


В исходном эксперименте `R²` увеличивался после возврата строк с нулевыми просмотрами. Но здесь важно помнить: после изменения набора данных немного меняется и состав train/test выборок. Поэтому разницу нельзя однозначно объяснять только наличием нулевых значений.


## Задание 3. Классификация

Определим две самые распространённые категории, построим логистическую регрессию и сравним разные наборы признаков.

Поскольку классы представлены не полностью одинаково, при разбиении используем `stratify=y`. В качестве основной метрики, кроме `accuracy`, дополнительно используем `macro F1-score`, который лучше учитывает качество классификации обоих классов.


In [ ]:
# Две самые распространённые категории

top_2_categories = df['category'].value_counts().head(2)
top_categories = top_2_categories.index

top_2_categories


In [ ]:
# Оставляем две категории и нужные признаки

df_class = df[df['category'].isin(top_categories)][
    ['video views', 'subscribers', 'uploads', 'category']
].copy()

# Убираем строки с нулевыми просмотрами
df_class = df_class[df_class['video views'] != 0]

print('Размер выборки:', df_class.shape)
print()
print(df_class['category'].value_counts())


In [ ]:
# Кодируем целевую переменную

le = LabelEncoder()
df_class['category_encoded'] = le.fit_transform(df_class['category'])

df_class[['category', 'category_encoded']].drop_duplicates()


In [ ]:
X = df_class[['video views', 'subscribers', 'uploads']]
y = df_class['category_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=13,
    stratify=y
)

# Проверим, что соотношение классов сохранилось
class_balance = pd.DataFrame({
    'Все данные': y.value_counts(normalize=True).sort_index(),
    'Train': y_train.value_counts(normalize=True).sort_index(),
    'Test': y_test.value_counts(normalize=True).sort_index()
})

class_balance


Параметр `stratify=y` сохраняет примерно одинаковое соотношение классов в исходной, обучающей и тестовой выборках. Это делает сравнение моделей стабильнее.


In [ ]:
model_log = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

model_log.fit(X_train, y_train)

y_pred = model_log.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print('Accuracy:', round(accuracy, 3))
print('Macro F1-score:', round(f1, 3))


`Accuracy` оставляем для наглядности и соответствия заданию, но при наличии дисбаланса основное внимание лучше уделять `macro F1-score`: он отдельно учитывает качество распознавания каждого класса и затем усредняет результат.


### Изменится ли качество модели при выборе других признаков?


In [ ]:
alt_features = [
    'rank',
    'lowest_monthly_earnings',
    'uploads'
]

# Используем те же строки train/test, чтобы сравнение было честным
X_alt = df.loc[df_class.index, alt_features]

X_train_alt = X_alt.loc[X_train.index]
X_test_alt = X_alt.loc[X_test.index]

model_log_alt = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

model_log_alt.fit(X_train_alt, y_train)
y_pred_alt = model_log_alt.predict(X_test_alt)

accuracy_alt = accuracy_score(y_test, y_pred_alt)
f1_alt = f1_score(y_test, y_pred_alt, average='macro')

print('Исходные признаки:')
print('Accuracy =', round(accuracy, 3))
print('Macro F1 =', round(f1, 3))

print('\nАльтернативные признаки:')
print('Accuracy =', round(accuracy_alt, 3))
print('Macro F1 =', round(f1_alt, 3))


Сравнивать наборы признаков лучше сразу по двум метрикам. Если и `accuracy`, и `macro F1` снижаются, можно уверенно сказать, что альтернативный набор признаков хуже разделяет категории `Entertainment` и `Music`.


### Сравнение с baseline


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

dummy_accuracy = accuracy_score(y_test, dummy_pred)
dummy_f1 = f1_score(y_test, dummy_pred, average='macro')

print('Логистическая регрессия:')
print('Accuracy =', round(accuracy, 3))
print('Macro F1 =', round(f1, 3))

print('\nBaseline:')
print('Accuracy =', round(dummy_accuracy, 3))
print('Macro F1 =', round(dummy_f1, 3))


Baseline показывает, какого качества можно добиться почти без использования информации из признаков. Если логистическая регрессия превосходит его и по `accuracy`, и по `macro F1`, значит признаки действительно помогают различать классы.


## Итог

В работе был проведён EDA данных о популярных YouTube-каналах, проверены пропуски, выбросы и основные взаимосвязи. Для пропусков дополнительно использована библиотека `missingno`, а часть значений в логически связанных признаках восстанавливалась только там, где соответствие было однозначным.

EDA показал выраженную асимметрию многих числовых признаков. Поэтому для общей оценки взаимосвязей была использована ранговая корреляция Спирмена. Коэффициент Пирсона для просмотров и подписчиков также рассчитан, поскольку этого требует задание, но сравнение со Спирменом показывает, что выбросы заметно усиливают линейную корреляцию.

Для линейной регрессии кроме `R²` добавлена диагностика остатков: их распределение и Q-Q график позволяют проверить, насколько хорошо выполняются предпосылки модели.

В задаче классификации разбиение выполняется со `stratify=y`, чтобы сохранить соотношение классов. Вместе с `accuracy` используется `macro F1-score`, который лучше подходит для оценки модели при неравномерном распределении классов. Также модель сравнивается с альтернативным набором признаков и простой baseline-моделью.

В результате работа не только выполняет исходные пункты задания, но и учитывает предпосылки статистических методов и более корректную диагностику качества моделей.
